In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import MarianTokenizer, MarianMTModel

In [ ]:


MODEL = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = MarianTokenizer.from_pretrained(MODEL)
model = MarianMTModel.from_pretrained(MODEL)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
print(model.config)

MarianConfig {
  "activation_dropout": 0.0,
  "activation_function": "swish",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "MarianMTModel"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 512,
  "decoder_attention_heads": 8,
  "decoder_ffn_dim": 2048,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 61949,
  "decoder_vocab_size": 61950,
  "dropout": 0.1,
  "dtype": "float32",
  "encoder_attention_heads": 8,
  "encoder_ffn_dim": 2048,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 0,
  "extra_pos_embeddings": 61950,
  "forced_eos_token_id": 0,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_position_embeddings": 512,
  "model_type": "marian"

In [ ]:
#Embedding matrix is all the words in the vocab
embedding_matrix = model.model.shared.weight
print ("Rows: " , len(embedding_matrix))
print ("Columns: " , len(embedding_matrix[0]))

Rows:  61950
Columns:  512


In [ ]:
sentence = "The history of Bharat is glorious"
#sentence = "ThehistoryofBharatisglorious"

In [ ]:
tokens = tokenizer.tokenize(sentence)
print (tokens)

['▁The', '▁history', '▁of', '▁B', 'hara', 't', '▁is', '▁glorious']


In [ ]:
token_ids = tokenizer.encode(sentence)
print (token_ids)

[81, 1853, 8, 1102, 27522, 142, 23, 6010, 0]


In [ ]:
for pos, tid in enumerate(token_ids):
    tok = tokenizer.decode([tid])
    print(f"{pos:<10} {tok:<15} {tid:>8}")

0          The                   81
1          history             1853
2          of                     8
3          B                   1102
4          hara               27522
5          t                    142
6          is                    23
7          glorious            6010
8          </s>                   0


In [ ]:
#print(embedding_matrix[token_id])
print(len(embedding_matrix[81]))

512


In [ ]:
print(tokenizer.decode([81]))

The


In [ ]:
print(tokenizer.decode([464]))

करे


In [ ]:
seq_len = len(token_ids)
print (seq_len)


9


In [ ]:
d_model = model.config.d_model
print (d_model)

512


In [ ]:
embedding_matrix = model.model.shared.weight
token_ids_tensor = torch.tensor(token_ids)
with torch.no_grad():
    embeddings = embedding_matrix[token_ids_tensor]

print (embeddings)

tensor([[ 0.0098, -0.0283, -0.0276,  ..., -0.0286, -0.0032, -0.0039],
        [ 0.0090, -0.0445,  0.0500,  ..., -0.1467, -0.0560, -0.0727],
        [-0.0080, -0.0076, -0.0018,  ..., -0.0132, -0.0558, -0.0919],
        ...,
        [ 0.0251,  0.0144, -0.0349,  ..., -0.0279, -0.0551,  0.0065],
        [-0.0097, -0.0192,  0.0512,  ...,  0.0152, -0.0100,  0.0269],
        [-0.0073, -0.0396, -0.0231,  ..., -0.0883, -0.0598, -0.0282]])


In [ ]:
print(f"\nPOSITIONAL ENCODING  (seq_len={seq_len}, d_model={d_model})")
print("Formula: PE(pos, 2i) = sin(pos / 10000^(2i/d))")
print("         PE(pos, 2i+1) = cos(pos / 10000^(2i/d))")

PE = np.zeros((seq_len, d_model))
for pos in range(seq_len):
    for i in range(0, d_model, 2):
        PE[pos, i]   = np.sin(pos / (10000 ** (i / d_model)))
        #print (PE[pos, i])
        PE[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))
        #print (PE[pos, i+1])


tokens_with_eos = tokens + ["<eos>"]
print(f"{'Token':<15} {'PE dims 0-9'}")
print("-" * 80)
for pos, tok in enumerate(tokens_with_eos):
    vals = " ".join([f"{PE[pos,i]:>6.3f}" for i in range(10)])
    print(f"{tok:<15} {vals}")



POSITIONAL ENCODING  (seq_len=9, d_model=512)
Formula: PE(pos, 2i) = sin(pos / 10000^(2i/d))
         PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
Token           PE dims 0-9
--------------------------------------------------------------------------------
▁The             0.000  1.000  0.000  1.000  0.000  1.000  0.000  1.000  0.000  1.000
▁history         0.841  0.540  0.822  0.570  0.802  0.597  0.782  0.623  0.762  0.648
▁of              0.909 -0.416  0.936 -0.351  0.958 -0.286  0.975 -0.223  0.987 -0.160
▁B               0.141 -0.990  0.245 -0.970  0.343 -0.939  0.434 -0.901  0.517 -0.856
hara            -0.757 -0.654 -0.657 -0.754 -0.549 -0.836 -0.434 -0.901 -0.317 -0.949
t               -0.959  0.284 -0.994  0.111 -0.998 -0.059 -0.975 -0.222 -0.928 -0.373
▁is             -0.279  0.960 -0.475  0.880 -0.644  0.765 -0.781  0.624 -0.885  0.465
▁glorious        0.657  0.754  0.452  0.892  0.229  0.973  0.001  1.000 -0.220  0.976
<eos>            0.989 -0.146  0.991  0.136  0.917  0.398  0

In [ ]:
# Build PE (same code as before, no prints inside loop)
PE = np.zeros((seq_len, d_model))
for pos in range(seq_len):
    for i in range(0, d_model, 2):
        PE[pos, i]   = np.sin(pos / (10000 ** (i / d_model)))
        PE[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))

tokens_with_eos = tokens + ["<eos>"]

# Show dims 0,1,2,3 ... 510,511 for each token
df = pd.DataFrame(
    {
        "token": tokens_with_eos,
        "dim0 (sin)": PE[:, 0].round(3),
        "dim1 (cos)": PE[:, 1].round(3),
        "dim2 (sin)": PE[:, 2].round(3),
        "dim3 (cos)": PE[:, 3].round(3),
        "dim510(sin)": PE[:, 510].round(4),
        "dim511(cos)": PE[:, 511].round(4),
    }
)
print(df.to_string(index=False))

    token  dim0 (sin)  dim1 (cos)  dim2 (sin)  dim3 (cos)  dim510(sin)  dim511(cos)
     ▁The       0.000       1.000       0.000       1.000       0.0000          1.0
 ▁history       0.841       0.540       0.822       0.570       0.0001          1.0
      ▁of       0.909      -0.416       0.936      -0.351       0.0002          1.0
       ▁B       0.141      -0.990       0.245      -0.970       0.0003          1.0
     hara      -0.757      -0.654      -0.657      -0.754       0.0004          1.0
        t      -0.959       0.284      -0.994       0.111       0.0005          1.0
      ▁is      -0.279       0.960      -0.475       0.880       0.0006          1.0
▁glorious       0.657       0.754       0.452       0.892       0.0007          1.0
    <eos>       0.989      -0.146       0.991       0.136       0.0008          1.0


In [ ]:
# Step 2c: Final Input = Embedding + Positional Encoding
# Both are shape (9, 512) — element-wise addition
PE_tensor = torch.tensor(PE, dtype=torch.float32)
final_input = embeddings + PE_tensor   # shape: (9, 512)

print(f"Embedding shape : {embeddings.shape}")
print(f"PE shape        : {PE_tensor.shape}")
print(f"Final input shape: {final_input.shape}")

# Show first 5 dims for each token
print("\nToken | Embed[0] | PE[0]  | FinalInput[0]")
print("-" * 55)
tokens_with_eos = tokens + ["<eos>"]
for i, tok in enumerate(tokens_with_eos):
    e = embeddings[i, 0].item()
    p = PE_tensor[i, 0].item()
    f = final_input[i, 0].item()
    print(f"{tok:<10} {e:>8.4f}   {p:>6.4f}   {f:>8.4f}")

Embedding shape : torch.Size([9, 512])
PE shape        : torch.Size([9, 512])
Final input shape: torch.Size([9, 512])

Token | Embed[0] | PE[0]  | FinalInput[0]
-------------------------------------------------------
▁The         0.0098   0.0000     0.0098
▁history     0.0090   0.8415     0.8504
▁of         -0.0080   0.9093     0.9013
▁B          -0.0204   0.1411     0.1207
hara        -0.0233   -0.7568    -0.7801
t            0.0892   -0.9589    -0.8698
▁is          0.0251   -0.2794    -0.2543
▁glorious   -0.0097   0.6570     0.6473
<eos>       -0.0073   0.9894     0.9821
